# `sampler_ansatz_real_amplitudes_pauliz` - 5-subset training

Uses datasets form `/mnt/data02/mkordasz/circuits/FRQI/MNIST_Digits/`.

## Imports

In [1]:
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchinfo
from torch.utils import data as torch_data
from torchmetrics import Accuracy

import lightning as L
from lightning.pytorch.loggers import CSVLogger

from qiskit.circuit.library import real_amplitudes

import geqie_qml
from geqie_qml import UnitaryInputLayer, SamplerAnsatzLayer

print("geqie_qml:", geqie_qml.__file__)

geqie_qml: /home/ODM/rpotempa/geqie/geqie-qml/src/geqie_qml/__init__.py


## Configuration

In [ ]:
# --- data ---
IMAGE_SIZE = 16                # MNIST is resized to IMAGE_SIZE x IMAGE_SIZE before FRQI encoding
ENCODING = "frqi"
DATASET_SAMPLE_SIZE = -1      # training images to encode
TEST_SAMPLE_SIZE = -1
VAL_SPLIT = 0.2
BATCH_SIZE = 16
PRECOMPUTED_ZIP = Path("/mnt/data02/mkordasz/circuits/FRQI/MNIST_Digits/") / f"subset_1.zip"

# --- model ---
N_CLASSES = 10
N_LAYERS = 4                    # real_amplitudes repetitions

torch.manual_seed(42)
np.random.seed(42)

## Precompute GEQIE encodings

Encodes a small MNIST subset with FRQI and packages the matrices into a single zip archive
(`train/` and `test/` folders), the layout expected by `load_precomputed_zip_matrices`.
Re-runs are skipped once the archive exists.

In [3]:
def resize_dataset(data: torch.Tensor, size: int) -> np.ndarray:
    resized = torch.nn.functional.interpolate(
        data.unsqueeze(1).float(),
        size=(size, size),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1)
    return resized.numpy().astype(np.uint8)


def build_precomputed_zip(n_workers) -> None:
    if PRECOMPUTED_ZIP.exists():
        print(f"Reusing existing archive: {PRECOMPUTED_ZIP}")
        return

    mnist_train = torchvision.datasets.MNIST(root="./.data", train=True, download=True)
    mnist_test = torchvision.datasets.MNIST(root="./.data", train=False, download=True)

    train_data = resize_dataset(mnist_train.data[:DATASET_SAMPLE_SIZE], IMAGE_SIZE)
    train_labels = mnist_train.targets[:DATASET_SAMPLE_SIZE]
    test_data = resize_dataset(mnist_test.data[:TEST_SAMPLE_SIZE], IMAGE_SIZE)
    test_labels = mnist_test.targets[:TEST_SAMPLE_SIZE]

    stage_dir = PRECOMPUTED_ZIP.parent / PRECOMPUTED_ZIP.stem
    geqie_qml.compute_and_save_circuits(
        data=train_data,
        labels=train_labels,
        save_dir=str(stage_dir / "train"),
        geqie_encoding=ENCODING,
        encoding_params={},
        number_of_workers=n_workers,
    )
    geqie_qml.compute_and_save_circuits(
        data=test_data,
        labels=test_labels,
        save_dir=str(stage_dir / "test"),
        geqie_encoding=ENCODING,
        encoding_params={},
        number_of_workers=n_workers,
    )

    with zipfile.ZipFile(PRECOMPUTED_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for matrix_file in stage_dir.glob("*/*.npz"):
            archive.write(matrix_file, arcname=matrix_file.relative_to(stage_dir).as_posix())
    print(f"Wrote archive: {PRECOMPUTED_ZIP}")


build_precomputed_zip(n_workers=4)

Reusing existing archive: /mnt/data02/mkordasz/circuits/FRQI/MNIST_Digits/subset_1.zip


In [4]:
train_ds, test_ds = geqie_qml.load_precomputed_zip_matrices(str(PRECOMPUTED_ZIP))

# Infer the qubit count from an actual encoded matrix (2**n x 2**n).
sample_matrix, _ = train_ds[0]
N_QUBITS = int(round(np.log2(sample_matrix.shape[-1])))
print(f"Encoded qubits: {N_QUBITS}")

n_total = len(train_ds)
n_val = int(n_total * VAL_SPLIT)
n_train = n_total - n_val
train_subset, val_subset = torch_data.random_split(
    train_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)

train_loader = torch_data.DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = torch_data.DataLoader(val_subset, batch_size=BATCH_SIZE, num_workers=0)
test_loader = torch_data.DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=0)
print(f"Train: {n_train}, Val: {n_val}, Test: {len(test_ds)}")

Encoded qubits: 9
Train: 737, Val: 184, Test: 288


## Model definition

`UnitaryInputLayer` turns each precomputed unitary matrix into the encoded statevector `U|0>`;
`SamplerAnsatzLayer` then applies the trainable `real_amplitudes` ansatz and returns the full
`2**N_QUBITS` measurement probability distribution. A weight-free `PauliZExpectation` layer
collapses that distribution to the `N_QUBITS` single-qubit `<Z_i>` expectation values before the
classical head.

In [5]:
class PauliZExpectation(nn.Module):
    """Weight-free read-out: full 2**n distribution -> n single-qubit <Z_i> expectations.

    <Z_i> = sum_x p(x) * (+1 if bit i of x is 0 else -1) is a constant linear map (no trainable
    parameters), so the classical head shrinks to Linear(n, n_classes).
    """

    def __init__(self, num_qubits: int):
        super().__init__()
        states = torch.arange(2 ** num_qubits)
        bits = (states.unsqueeze(1) >> torch.arange(num_qubits)) & 1  # (2**n, n)
        self.register_buffer("signs", 1.0 - 2.0 * bits.float())        # +1 for |0>, -1 for |1>

    def forward(self, probs: torch.Tensor) -> torch.Tensor:
        return probs.to(self.signs.dtype) @ self.signs                 # (B, 2**n) -> (B, n)


ansatz = real_amplitudes(N_QUBITS, reps=N_LAYERS)

model = nn.Sequential(
    UnitaryInputLayer(N_QUBITS),                     # (B, dim, dim) complex -> (B, dim) complex
    SamplerAnsatzLayer(N_QUBITS, ansatz, seed=42),    # (B, dim) complex -> (B, dim) probs
    PauliZExpectation(N_QUBITS),                      # (B, 2**N_QUBITS) -> (B, N_QUBITS), 0 params
    nn.Linear(N_QUBITS, N_CLASSES),
    nn.LogSoftmax(dim=-1),
)
torchinfo.summary(model)

Layer (type:depth-idx)                   Param #
Sequential                               --
├─UnitaryInputLayer: 1-1                 --
├─SamplerAnsatzLayer: 1-2                45
├─PauliZExpectation: 1-3                 --
├─Linear: 1-4                            100
├─LogSoftmax: 1-5                        --
Total params: 145
Trainable params: 145
Non-trainable params: 0

## Training with Lightning

In [6]:
class SamplerAnsatzModule(L.LightningModule):
    """Lightning wrapper; the model ends with LogSoftmax, so the loss is NLL."""

    def __init__(self, model: nn.Module, n_classes: int, lr: float = 1e-1):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.train_acc = Accuracy(task="multiclass", num_classes=n_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=n_classes)
        self.test_acc = Accuracy(task="multiclass", num_classes=n_classes)

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch):
        x, y = batch
        log_probs = self(x)
        loss = F.nll_loss(log_probs, y)
        preds = log_probs.argmax(dim=-1)
        return loss, preds, y

    def training_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.train_acc(preds, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_acc",  self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("lr", self.trainer.optimizers[0].param_groups[0]["lr"], on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.val_acc(preds, y)
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
        self.log("val_acc", self.val_acc, on_epoch=True, prog_bar=True)

    def test_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.test_acc(preds, y)
        self.log("test_loss", loss, on_epoch=True, prog_bar=True)
        self.log("test_acc", self.test_acc, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }

In [7]:
module = SamplerAnsatzModule(model, n_classes=N_CLASSES, lr=0.1)
logger = CSVLogger(save_dir="lightning_logs", name="sampler_ansatz_real_amplitudes_pauliz")

trainer = L.Trainer(
    max_epochs=50,
    accelerator="cpu",
    devices=1,
    logger=logger,
    log_every_n_steps=1,
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    callbacks=[
        L.pytorch.callbacks.EarlyStopping(monitor="val_loss", patience=10, mode="min"),
        L.pytorch.callbacks.TQDMProgressBar(leave=True),
    ],
)

# SamplerAnsatzLayer runs pure NumPy/PyTorch autograd in-process; no process pool needed.
trainer.fit(module, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name      | Type               | Params | Mode  | FLOPs
-----------------------------------------------------------------
0 | model     | Sequential         | 145    | train | 0    
1 | train_acc | MulticlassAccuracy | 0      | train | 0    
2 | val_acc   | MulticlassAccuracy | 0      | train | 0    
3 | test_acc  | MulticlassAccuracy | 0      | train | 0    
-----------------------------------------------------------------
145       Trainable params
0         Non-trainable params

Training: |                                                                                                                                                                               | 0/? [00:00<?, ?it/s]
Epoch 42: 100%|██████████████████████████████████████████████████████████████████████████| 47/47 [00:58<00:00,  0.80it/s, v_num=1, val_loss=1.160, val_acc=0.603, train_loss=0.824, train_acc=0.730, lr=0.00312]


In [8]:
test_metrics = trainer.test(module, dataloaders=test_loader)
test_metrics

/home/ODM/rpotempa/geqie/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:05<00:00,  3.17it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc                  0.5625
        test_loss           1.1896411180496216
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 1.1896411180496216, 'test_acc': 0.5625}]

## Training curves

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TRAIN_COLOR = "#1f77b4"
VAL_COLOR = "#ff7f0e"

metrics = pd.read_csv(f"{trainer.logger.log_dir}/metrics.csv")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss", "Accuracy"))
for cols, col_idx in [(["train_loss", "val_loss"], 1), (["train_acc", "val_acc"], 2)]:
    for col in cols:
        if col not in metrics:
            continue
        subset = metrics.dropna(subset=[col])
        color = TRAIN_COLOR if col.startswith("train") else VAL_COLOR
        fig.add_trace(
            go.Scatter(
                x=subset["epoch"],
                y=subset[col],
                mode="lines+markers",
                name=col,
                line=dict(color=color),
                marker=dict(color=color),
            ),
            row=1,
            col=col_idx,
        )

fig.update_xaxes(title_text="Epoch")
fig.update_layout(height=400, width=900, title_text="sampler_ansatz_real_amplitudes_pauliz")
fig.show()